---
title: "Parallel Research and Fan-In"
draft: true
categories: [agents, workflows, langgraph]
---


Evidence collection is a map-reduce workflow: the planner emits bounded tasks, each branch owns its input, and synthesis waits for an accountable fan-in. `Send` expresses the dynamic branch count without sharing mutable branch state.

## `Send` maps tasks into isolated state

This small graph exposes the teaching-critical primitive directly. The reducer only appends branch results; completeness and duplicate checks belong in the join node.


In [1]:
import operator
from typing import Annotated, TypedDict
from langgraph.graph import END, START, StateGraph
from langgraph.types import Send

class MapState(TypedDict):
    tasks: list[dict[str, int | str]]
    results: Annotated[list[dict[str, int | str]], operator.add]
    summary: dict[str, int]

class BranchState(TypedDict):
    task: dict[str, int | str]

def make_tasks(state: MapState):
    return {"tasks": [{"id": "security", "latency": 30}, {"id": "performance", "latency": 20}, {"id": "operations", "latency": 10}]}
def dispatch(state: MapState):
    return [Send("collect", {"task": task}) for task in state["tasks"]]
def collect(state: BranchState):
    return {"results": [{"task_id": state["task"]["id"], "latency": state["task"]["latency"]}]}
def join(state: MapState):
    ids = [str(row["task_id"]) for row in state["results"]]
    assert len(ids) == len(set(ids)) == len(state["tasks"])
    return {"summary": {"branches": len(ids), "critical_path_ms": max(int(r["latency"]) for r in state["results"])}}

builder = StateGraph(MapState)
builder.add_node("make_tasks", make_tasks)
builder.add_node("collect", collect)
builder.add_node("join", join)
builder.add_edge(START, "make_tasks")
builder.add_conditional_edges("make_tasks", dispatch, ["collect"])
builder.add_edge("collect", "join")
builder.add_edge("join", END)
map_graph = builder.compile()
map_graph.invoke({"tasks": [], "results": [], "summary": {}})


{'tasks': [{'id': 'security', 'latency': 30},
  {'id': 'performance', 'latency': 20},
  {'id': 'operations', 'latency': 10}],
 'results': [{'task_id': 'security', 'latency': 30},
  {'task_id': 'performance', 'latency': 20},
  {'task_id': 'operations', 'latency': 10}],
 'summary': {'branches': 3, 'critical_path_ms': 30}}

The three branches report sixty milliseconds of work but a thirty-millisecond simulated critical path. The join, rather than the reducer, decides whether the research set is complete.

## Parallelism changes cost and recovery

Run the package graph in parallel and sequential modes, then remove one branch. The same evidence contract applies to both topologies.


In [2]:
from evidence_brief.schemas import FaultPlan
from evidence_brief.workflow import run_fixture

parallel = run_fixture("conflict-01", variant="full")
sequential = run_fixture("conflict-01", variant="sequential")
gap = run_fixture("conflict-01", faults=FaultPlan(missing_task_id="operations"))

for name, state in (("parallel", parallel), ("sequential", sequential), ("missing branch", gap)):
    print(name, {
        "status": state["status"],
        "branches": len({row["task_id"] for row in state["branch_results"]}),
        "latency_ms": state["run_metrics"]["simulated_latency_ms"],
        "cost_units": state["run_metrics"]["cost_units"],
        "reason": state["terminal_reason"],
    })

def route_gap(missing: list[str], attempts: int, policy: str) -> str:
    if not missing:
        return "reconcile"
    if policy == "replan" and attempts < 1:
        return "plan"
    return "complete_with_explicit_gap"

print({
    "strict_policy": route_gap(["operations"], 0, "replan"),
    "deadline_policy": route_gap(["operations"], 1, "continue"),
})
assert parallel["run_metrics"]["simulated_latency_ms"] < sequential["run_metrics"]["simulated_latency_ms"]
assert gap["status"] == "failed" and "operations" in gap["terminal_reason"]
assert route_gap(["operations"], 0, "replan") == "plan"
assert route_gap(["operations"], 1, "continue") == "complete_with_explicit_gap"


parallel {'status': 'complete', 'branches': 3, 'latency_ms': 30, 'cost_units': 14, 'reason': 'completion contract satisfied'}
sequential {'status': 'complete', 'branches': 3, 'latency_ms': 60, 'cost_units': 14, 'reason': 'completion contract satisfied'}
missing branch {'status': 'failed', 'branches': 3, 'latency_ms': 30, 'cost_units': 10, 'reason': 'missing branches: operations'}
{'strict_policy': 'plan', 'deadline_policy': 'complete_with_explicit_gap'}


Parallelism reduces the critical path without changing the number of tool calls. It also forces the workflow to account for every scheduled task. A partial result is either an explicit gap or a reason to replan; it is never silently treated as complete.
